In [41]:
!pip install pandas
!pip install yfinance
!pip install scipy
!pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=b1e257bd9c0a28e352cef2402f7d9302de91c03cf69bb7ab2d54a6e8acb9f576
  Stored in directory: /Users/aryamangoenka/Library/Caches/pip/wheels/61/d8/66/8018676d483fa5edb5337a7a25ee8c029ac255be25f90f3cd5
Successfully built ta


In [42]:
import pandas as pd
import yfinance as yf
import numpy as np
from scipy import stats
from datetime import datetime, timedelta
import ta

In [32]:
df = pd.read_csv('nasdaq_stocks.csv')  # Ensure the file contains a 'Symbol' column
nasdaq_stocks = df.sample(n=250, random_state=None)
print(nasdaq_stocks)

     Symbol                                      Security Name
2682   MCHI                             iShares MSCI China ETF
2762   MKZR      MacKenzie Realty Capital, Inc. - Common Stock
3650   RFEU          First Trust RiverFront Dynamic Europe ETF
3809    SDA     SunCar Technology Group Inc. - Ordinary Shares
1653   FRSX  Foresight Autonomous Holdings Ltd. - American ...
...     ...                                                ...
4173   TCOM  Trip.com Group Limited - American Depositary S...
4256   TNXP  Tonix Pharmaceuticals Holding Corp. - Common S...
3348   PLAO  Patria Latin American Opportunity Acquisition ...
3169   ONMD              OneMedNet Corp - Class A Common Stock
1537    FEM         First Trust Emerging Markets AlphaDEX Fund

[250 rows x 2 columns]


In [33]:
def calculate_rs_ratings(df, symbol_column='Symbol', rs_period=252):
    """
    Calculates the Relative Strength (RS) rating of stocks relative to the S&P 500.

    Parameters:
    - df (pd.DataFrame): DataFrame containing stock symbols.
    - symbol_column (str): Name of the column containing stock symbols.
    - rs_period (int): Number of trading days to calculate RS (default is ~12 months).

    Returns:
    - pd.DataFrame: Original DataFrame with an added 'RS_Rating' column.
    """
    
    # Define the benchmark
    benchmark_symbol = '^GSPC'  # S&P 500 index symbol in Yahoo Finance

    # Calculate the start and end dates
    end_date = datetime.today()
    start_date = end_date - timedelta(days=rs_period*1.5)  # Extra days to account for weekends/holidays

    # Fetch benchmark data
    print("Fetching S&P 500 data...")
    benchmark_data = yf.download(benchmark_symbol, start=start_date, end=end_date, progress=False)
    if benchmark_data.empty:
        raise ValueError("Failed to fetch S&P 500 data.")
    benchmark_data = benchmark_data['Adj Close'].dropna()

    # Calculate benchmark returns
    benchmark_start = benchmark_data.iloc[0]
    benchmark_end = benchmark_data.iloc[-1]
    benchmark_return = (benchmark_end / benchmark_start - 1) * 100  # Percentage

    print(f"S&P 500 return over the period: {benchmark_return:.2f}%")

    # Lists to store results
    symbols = []
    rs_returns = []

    # Iterate over each symbol to calculate returns
    for symbol in df[symbol_column]:
        try:
            print(f"Fetching data for {symbol}...")
            stock_data = yf.download(symbol, start=start_date, end=end_date, progress=False)
            if stock_data.empty:
                print(f"Warning: No data fetched for {symbol}. Skipping.")
                continue
            stock_data = stock_data['Adj Close'].dropna()

            # Ensure we have enough data
            if len(stock_data) < rs_period:
                print(f"Warning: Not enough data for {symbol}. Skipping.")
                continue

            # Calculate stock returns
            stock_start = stock_data.iloc[0]
            stock_end = stock_data.iloc[-1]
            stock_return = (stock_end / stock_start - 1) * 100  # Percentage

            # Relative Strength
            rs = stock_return - benchmark_return

            symbols.append(symbol)
            rs_returns.append(rs)

            print(f"{symbol}: Stock Return = {stock_return:.2f}%, RS = {rs:.2f}%")
        
        except Exception as e:
            print(f"Error processing {symbol}: {e}")
            continue

    # Create a DataFrame for RS
    rs_df = pd.DataFrame({
        'Symbol': symbols,
        'RS': rs_returns
    })

    # Calculate RS Rating as percentile
    rs_df['RS_Rating'] = rs_df['RS'].rank(pct=True) * 100  # Percentile (0-100)

    # Merge RS_Rating back to the original DataFrame
    result_df = df.merge(rs_df[['Symbol', 'RS_Rating']], on='Symbol', how='left')

    return result_df

In [53]:
calculate_rs_ratings(nasdaq_stocks)

Fetching S&P 500 data...
S&P 500 return over the period: 32.66%
Fetching data for MCHI...
MCHI: Stock Return = 10.12%, RS = -22.53%
Fetching data for MKZR...
Fetching data for RFEU...
RFEU: Stock Return = 5.79%, RS = -26.86%
Fetching data for SDA...
SDA: Stock Return = 60.06%, RS = 27.41%
Fetching data for FRSX...
FRSX: Stock Return = -62.81%, RS = -95.46%
Fetching data for BSRR...
BSRR: Stock Return = 74.03%, RS = 41.37%
Fetching data for HIVE...
HIVE: Stock Return = 38.31%, RS = 5.65%
Fetching data for FSV...
FSV: Stock Return = 26.27%, RS = -6.39%
Fetching data for CANC...
CANC: Stock Return = 21.04%, RS = -11.61%
Fetching data for ONMDW...
Fetching data for BCAN...
BCAN: Stock Return = -99.76%, RS = -132.41%
Fetching data for NXLIW...
Fetching data for REYN...
REYN: Stock Return = 8.05%, RS = -24.61%
Fetching data for JG...
JG: Stock Return = 271.82%, RS = 239.16%
Fetching data for IGTAR...
Fetching data for CMPO...
CMPO: Stock Return = 212.85%, RS = 180.19%
Fetching data for GH...

UBX: Stock Return = -38.61%, RS = -71.27%
Fetching data for GLLIU...
GLLIU: Stock Return = 6.92%, RS = -25.74%
Fetching data for GOCO...
GOCO: Stock Return = -1.84%, RS = -34.50%
Fetching data for ALVO...
ALVO: Stock Return = 28.56%, RS = -4.10%
Fetching data for PPTA...
PPTA: Stock Return = 198.75%, RS = 166.09%
Fetching data for VEON...
VEON: Stock Return = 63.14%, RS = 30.49%
Fetching data for AZTA...
AZTA: Stock Return = -17.50%, RS = -50.15%
Fetching data for FPXI...
FPXI: Stock Return = 23.95%, RS = -8.71%
Fetching data for SMCX...
Fetching data for CDROW...
Fetching data for SANW...
SANW: Stock Return = -42.91%, RS = -75.57%
Fetching data for RUMBW...
Fetching data for HEPA...
HEPA: Stock Return = -78.79%, RS = -111.45%
Fetching data for INDP...
INDP: Stock Return = -49.25%, RS = -81.91%
Fetching data for HLXB...
Fetching data for CSPI...
CSPI: Stock Return = 31.99%, RS = -0.67%
Fetching data for ASNS...
ASNS: Stock Return = 9.82%, RS = -22.83%
Fetching data for BANF...
BANF: St

NXTG: Stock Return = 20.78%, RS = -11.88%
Fetching data for AVS...
Fetching data for OPINL...
OPINL: Stock Return = 2.05%, RS = -30.61%
Fetching data for DIOD...
DIOD: Stock Return = -8.33%, RS = -40.99%
Fetching data for AVTE...
AVTE: Stock Return = -81.87%, RS = -114.53%
Fetching data for HTZWW...
Fetching data for TCOM...
TCOM: Stock Return = 76.30%, RS = 43.65%
Fetching data for TNXP...
TNXP: Stock Return = -98.76%, RS = -131.42%
Fetching data for PLAO...
PLAO: Stock Return = 5.34%, RS = -27.31%
Fetching data for ONMD...
ONMD: Stock Return = -80.17%, RS = -112.82%
Fetching data for FEM...
FEM: Stock Return = 8.08%, RS = -24.58%


,Symbol,Security Name,RS_Rating
0,MCHI,iShares MSCI China ETF,46.231156
1,MKZR,"MacKenzie Realty Capital, Inc. - Common Stock",NaN
2,RFEU,First Trust RiverFront Dynamic Europe ETF,39.698492
3,SDA,SunCar Technology Group Inc. - Ordinary Shares,79.396985
4,FRSX,Foresight Autonomous Holdings Ltd. - American ...,11.557789
...,...,...,...
245,TCOM,Trip.com Group Limited - American Depositary S...,83.919598
246,TNXP,Tonix Pharmaceuticals Holding Corp. - Common S...,2.512563
247,PLAO,Patria Latin American Opportunity Acquisition ...,38.190955
248,ONMD,OneMedNet Corp - Class A Common Stock,7.537688


In [54]:
def calculate_technical_indicators_with_ta(df, symbol_column='Symbol'):
    """
    Calculates technical indicators for each stock using the 'ta' library.
    
    Indicators:
    - 50 Day Moving Average
    - 150 Day Moving Average
    - 200 Day Moving Average
    - 52 Week Low
    - 52 Week High
    """
    # Define required periods
    ma_periods = [50, 150, 200]
    weeks_52 = 252  # Approximate number of trading days in 52 weeks

    # Calculate the start date to fetch sufficient data
    end_date = datetime.today()
    start_date = end_date - timedelta(days=weeks_52 * 2)  # Buffer for non-trading days

    # Lists to store results
    symbols = []
    ma_50_list = []
    ma_150_list = []
    ma_200_list = []
    low_52w_list = []
    high_52w_list = []

    # Iterate over each symbol to calculate indicators
    for symbol in df[symbol_column]:
        try:
            print(f"Fetching data for {symbol} for technical indicators...")
            stock_data = yf.download(symbol, start=start_date, end=end_date, progress=False)
            if stock_data.empty:
                print(f"Warning: No data fetched for {symbol}. Skipping.")
                continue
            adj_close = stock_data['Adj Close'].dropna()

            # Ensure we have enough data
            if len(adj_close) < max(ma_periods + [weeks_52]):
                print(f"Warning: Not enough data for {symbol}. Skipping.")
                continue

            # Calculate Moving Averages
            ma_50 = adj_close.rolling(window=50).mean().iloc[-1]
            ma_150 = adj_close.rolling(window=150).mean().iloc[-1]
            ma_200 = adj_close.rolling(window=200).mean().iloc[-1]

            # Calculate 52 Week Low and High
            low_52w = adj_close.rolling(window=weeks_52).min().iloc[-1]
            high_52w = adj_close.rolling(window=weeks_52).max().iloc[-1]

            # Append results
            symbols.append(symbol)
            ma_50_list.append(ma_50)
            ma_150_list.append(ma_150)
            ma_200_list.append(ma_200)
            low_52w_list.append(low_52w)
            high_52w_list.append(high_52w)

            print(f"{symbol}: 50 MA = {ma_50:.2f}, 150 MA = {ma_150:.2f}, 200 MA = {ma_200:.2f}, "
                  f"52W Low = {low_52w:.2f}, 52W High = {high_52w:.2f}")

        except Exception as e:
            print(f"Error processing {symbol}: {e}")
            continue

    # Create a DataFrame for Indicators
    indicators_df = pd.DataFrame({
        'Stock': symbols,
        'SMA_50': ma_50_list,
        'SMA_150': ma_150_list,
        'SMA_200': ma_200_list,
#         '52 Week Low': low_52w_list,
#         '52 Week High': high_52w_list
    })

    return indicators_df

In [58]:
nasdaq_stocks = calculate_technical_indicators_with_ta(nasdaq_stocks)

Fetching data for MCHI for technical indicators...
MCHI: 50 MA = 49.75, 150 MA = 45.12, 200 MA = 43.75, 52W Low = 35.88, 52W High = 59.67
Fetching data for MKZR for technical indicators...
Fetching data for RFEU for technical indicators...
RFEU: 50 MA = 64.47, 150 MA = 65.04, 200 MA = 64.59, 52W Low = 59.23, 52W High = 68.33
Fetching data for SDA for technical indicators...
SDA: 50 MA = 9.98, 150 MA = 9.22, 200 MA = 8.64, 52W Low = 6.19, 52W High = 11.41
Fetching data for FRSX for technical indicators...
FRSX: 50 MA = 0.66, 150 MA = 0.86, 200 MA = 0.92, 52W Low = 0.59, 52W High = 2.02
Fetching data for BSRR for technical indicators...
BSRR: 50 MA = 29.55, 150 MA = 26.18, 200 MA = 24.20, 52W Low = 17.22, 52W High = 34.59
Fetching data for HIVE for technical indicators...
HIVE: 50 MA = 3.74, 150 MA = 3.27, 200 MA = 3.28, 52W Low = 2.27, 52W High = 5.72
Fetching data for FSV for technical indicators...
FSV: 50 MA = 186.02, 150 MA = 169.46, 200 MA = 167.33, 52W Low = 141.40, 52W High = 197

RDUS: 50 MA = 18.07, 150 MA = 16.66, 200 MA = 17.34, 52W Low = 13.54, 52W High = 30.43
Fetching data for CTKB for technical indicators...
CTKB: 50 MA = 5.62, 150 MA = 5.75, 200 MA = 6.06, 52W Low = 4.69, 52W High = 9.51
Fetching data for EJH for technical indicators...
EJH: 50 MA = 0.92, 150 MA = 5.12, 200 MA = 8.53, 52W Low = 0.71, 52W High = 219.00
Fetching data for SOXQ for technical indicators...
SOXQ: 50 MA = 40.29, 150 MA = 40.36, 200 MA = 39.60, 52W Low = 28.57, 52W High = 46.46
Fetching data for LPAA for technical indicators...
Fetching data for BNGO for technical indicators...
BNGO: 50 MA = 0.33, 150 MA = 0.57, 200 MA = 0.69, 52W Low = 0.21, 52W High = 2.10
Fetching data for ERNA for technical indicators...
ERNA: 50 MA = 0.96, 150 MA = 1.55, 200 MA = 1.68, 52W Low = 0.43, 52W High = 2.47
Fetching data for RVSNW for technical indicators...
Fetching data for ICCM for technical indicators...
ICCM: 50 MA = 0.65, 150 MA = 0.75, 200 MA = 0.88, 52W Low = 0.53, 52W High = 1.48
Fetchin

INDP: 50 MA = 1.23, 150 MA = 1.77, 200 MA = 1.88, 52W Low = 1.00, 52W High = 2.80
Fetching data for HLXB for technical indicators...
Fetching data for CSPI for technical indicators...
CSPI: 50 MA = 13.22, 150 MA = 13.82, 200 MA = 15.13, 52W Low = 8.29, 52W High = 27.84
Fetching data for ASNS for technical indicators...
ASNS: 50 MA = 1.36, 150 MA = 1.34, 200 MA = 1.25, 52W Low = 0.44, 52W High = 3.72
Fetching data for BANF for technical indicators...
BANF: 50 MA = 112.94, 150 MA = 100.68, 200 MA = 96.93, 52W Low = 80.51, 52W High = 128.09
Fetching data for MYCK for technical indicators...
Fetching data for CGEN for technical indicators...
CGEN: 50 MA = 1.67, 150 MA = 1.84, 200 MA = 1.99, 52W Low = 0.66, 52W High = 2.95
Fetching data for FOLD for technical indicators...
FOLD: 50 MA = 10.73, 150 MA = 10.56, 200 MA = 10.90, 52W Low = 9.04, 52W High = 14.52
Fetching data for RYTM for technical indicators...
RYTM: 50 MA = 54.05, 150 MA = 47.46, 200 MA = 46.03, 52W Low = 33.43, 52W High = 67.

OMGA: 50 MA = 1.08, 150 MA = 1.61, 200 MA = 2.05, 52W Low = 0.76, 52W High = 5.32
Fetching data for AVDL for technical indicators...
AVDL: 50 MA = 12.99, 150 MA = 14.80, 200 MA = 15.07, 52W Low = 10.72, 52W High = 18.82
Fetching data for KXIN for technical indicators...
KXIN: 50 MA = 6.97, 150 MA = 7.14, 200 MA = 8.29, 52W Low = 1.96, 52W High = 101.40
Fetching data for IBTF for technical indicators...
IBTF: 50 MA = 23.28, 150 MA = 23.01, 200 MA = 22.89, 52W Low = 22.24, 52W High = 23.36
Fetching data for ATLC for technical indicators...
ATLC: 50 MA = 40.74, 150 MA = 33.70, 200 MA = 32.68, 52W Low = 23.45, 52W High = 58.73
Fetching data for TETEW for technical indicators...
Fetching data for RGTI for technical indicators...
RGTI: 50 MA = 1.23, 150 MA = 1.08, 200 MA = 1.20, 52W Low = 0.69, 52W High = 3.05
Fetching data for INDB for technical indicators...
INDB: 50 MA = 64.69, 150 MA = 57.70, 200 MA = 55.65, 52W Low = 45.46, 52W High = 74.97
Fetching data for GP for technical indicators.

In [59]:
# yf.pdr_override() 
# start =dt.datetime(2017,12,1)
# now = dt.datetime.now()
# final_df.DataFrame()

# for i in nasdaq_stocks.index:
#     stock=str(nasdaq_stocks["Symbol"][i])
#     #RS_Rating=nasdaq_stocks["RS Rating"][i]

#     try:
#         df = pdr.get_data_yahoo(stock, start, now)

#         smaUsed=[50,150,200]
#         for x in smaUsed:
#             sma=x
#             df["SMA_"+str(sma)]=round(df.iloc[:,4].rolling(window=sma).mean(),2)


#         currentClose=df["Adj Close"][-1]
#         moving_average_50=df["SMA_50"][-1]
#         moving_average_150=df["SMA_150"][-1]
#         moving_average_200=df["SMA_200"][-1]
#         low_of_52week=min(df["Adj Close"][-260:])
#         high_of_52week=max(df["Adj Close"][-260:])
        
#         try:
#             moving_average_200_20 = df["SMA_200"][-20]

#         except Exception:
#             moving_average_200_20=0

#         #Condition 1: Current Price > 150 SMA and > 200 SMA
#         if(currentClose>moving_average_150>moving_average_200):
#             cond_1=True
#         else:
#             cond_1=False
            
#         #Condition 2: 150 SMA and > 200 SMA
#         if(moving_average_150>moving_average_200):
#             cond_2=True
#         else:
#             cond_2=False
            
#         #Condition 3: 200 SMA trending up for at least 1 month (ideally 4-5 months)
#         if(moving_average_200>moving_average_200_20):
#             cond_3=True
#         else:
#             cond_3=False
            
#         #Condition 4: 50 SMA> 150 SMA and 50 SMA> 200 SMA
#         if(moving_average_50>moving_average_150>moving_average_200):
#             #print("Condition 4 met")
#             cond_4=True
#         else:
#             #print("Condition 4 not met")
#             cond_4=False
            
#         #Condition 5: Current Price > 50 SMA
#         if(currentClose>moving_average_50):
#             cond_5=True
#         else:
#             cond_5=False
            
#         #Condition 6: Current Price is at least 30% above 52 week low (Many of the best are up 100-300% before coming out of consolidation)
#         if(currentClose>=(1.3*low_of_52week)):
#             cond_6=True
#         else:
#             cond_6=False
            
#         #Condition 7: Current Price is within 25% of 52 week high
#         if(currentClose>=(.75*high_of_52week)):
#             cond_7=True
#         else:
#             cond_7=False
            
#         #Condition 8: IBD RS rating >70 and the higher the better
#         #if(RS_Rating>70):
#         #cond_8=True
#         #else:
#         cond_8=False

#         if(cond_1 and cond_2 and cond_3 and cond_4 and cond_5 and cond_6 and cond_7 and cond_8):
#             final_df.append({'Stock': stock, "RS_Rating": RS_Rating, "50 Day MA": moving_average_50, "150 Day Ma": moving_average_150, "200 Day MA": moving_average_200, "52 Week Low": low_of_52week, "52 week High": high_of_52week}, ignore_index=True)
# #         except Exception:
# #             print("No data on "+stock)

# # print(final_df)



In [60]:
nasdaq_stocks

,Stock,SMA_50,SMA_150,SMA_200
0,MCHI,49.747400,45.121796,43.751242
1,RFEU,64.467372,65.039977,64.590370
2,SDA,9.978200,9.223533,8.643750
3,FRSX,0.658580,0.861107,0.915800
4,BSRR,29.554496,26.175038,24.202738
...,...,...,...,...
194,TCOM,62.332000,52.830000,51.105950
195,TNXP,0.154600,1.392667,3.254100
196,PLAO,11.616000,11.520100,11.460280
197,ONMD,0.762940,0.955800,0.905180


In [64]:
# List to collect rows before creating the final DataFrame
rows_list = []

# Iterate over each stock in nasdaq_stocks
for i in nasdaq_stocks.index:
    stock = str(nasdaq_stocks["Stock"][i])

    try:
        print(f"Fetching data for {stock}...")
        
#         # Get 52-week low and high (assuming 260 trading days)
#         if len(df) >= 260:
#             low_of_52week = df["Adj Close"].iloc[-260:].min()
#             high_of_52week = df["Adj Close"].iloc[-260:].max()
#         else:
#             print(f"Not enough data for {stock} to calculate 52-week low/high. Skipping.")
#             continue
        moving_average_50 = nasdaq_stocks["SMA_50"][i]
        moving_average_150 = nasdaq_stocks["SMA_150"][i]
        moving_average_200 = nasdaq_stocks["SMA_200"][i]
#         # Attempt to get SMA_200 from 20 days ago
#         if len(df) >= 20:
#             moving_average_200_20 = df["SMA_200"].iloc[-20]
#         else:
#             moving_average_200_20 = 0  # Default value if not enough data

        # Define conditions
#         cond_1 = currentClose > moving_average_150 > moving_average_200
        cond_2 = moving_average_150 > moving_average_200
        #cond_3 = moving_average_200 > moving_average_200
        cond_4 = moving_average_50 > moving_average_150
#         cond_5 = currentClose > moving_average_50
#         cond_6 = currentClose >= (1.3 * low_of_52week)
#         cond_7 = currentClose >= (0.75 * high_of_52week)
#         cond_8 = RS_Rating > 70  # Ensure RS_Rating is properly defined

        # Check if all conditions are met
        #if all([cond_1, cond_2, cond_3, cond_4, cond_5, cond_6, cond_7, cond_8]):
        if all([cond_2, cond_4]):
            row = {
                'Stock': stock,
#                 'RS_Rating': RS_Rating,
                '50 Day MA': moving_average_50,
                '150 Day MA': moving_average_150,
                '200 Day MA': moving_average_200,
#                 '52 Week Low': low_of_52week,
#                 '52 week High': high_of_52week
            }
            rows_list.append(row)
            print(f"{stock} meets all conditions. Added to final_df.")
        else:
            print(f"{stock} does not meet all conditions.")

    except Exception as e:
        print(f"Error processing {stock}: {e}")

# Create the final DataFrame from the collected rows
if rows_list:
    final_df = pd.DataFrame(rows_list)
    print("\nFinal DataFrame with Stocks Meeting All Conditions:")
    print(final_df)
else:
    print("No stocks met all conditions.")


Fetching data for MCHI...
MCHI meets all conditions. Added to final_df.
Fetching data for RFEU...
RFEU does not meet all conditions.
Fetching data for SDA...
SDA meets all conditions. Added to final_df.
Fetching data for FRSX...
FRSX does not meet all conditions.
Fetching data for BSRR...
BSRR meets all conditions. Added to final_df.
Fetching data for HIVE...
HIVE does not meet all conditions.
Fetching data for FSV...
FSV meets all conditions. Added to final_df.
Fetching data for CANC...
CANC does not meet all conditions.
Fetching data for BCAN...
BCAN does not meet all conditions.
Fetching data for REYN...
REYN meets all conditions. Added to final_df.
Fetching data for JG...
JG meets all conditions. Added to final_df.
Fetching data for CMPO...
CMPO meets all conditions. Added to final_df.
Fetching data for GH...
GH does not meet all conditions.
Fetching data for UHG...
UHG does not meet all conditions.
Fetching data for CWCO...
CWCO does not meet all conditions.
Fetching data for GNSS

In [65]:
print(final_df)

    Stock   50 Day MA  150 Day MA  200 Day MA
0    MCHI   49.747400   45.121796   43.751242
1     SDA    9.978200    9.223533    8.643750
2    BSRR   29.554496   26.175038   24.202738
3     FSV  186.016313  169.457412  167.334401
4    REYN   28.864341   28.693898   28.525766
..    ...         ...         ...         ...
83  BWBBP   19.640148   18.472234   18.095339
84   NXTG   86.236236   82.944483   81.435611
85  OPINL   12.767989   11.166599   10.825397
86   TCOM   62.332000   52.830000   51.105950
87   PLAO   11.616000   11.520100   11.460280

[88 rows x 4 columns]
